# Sales Performance Analysis — Sample Superstore

**Portfolio project | Python, pandas, Matplotlib, Seaborn, Power BI**

## Business objective
Analyze historical retail orders to understand sales, profit, product/category performance, regional performance, and trends over time. Findings should be based on the actual dataset; do not assume high sales always mean high profit.

## How to run
1. Download a permitted Sample Superstore CSV and place it in `data/raw/`.
2. Update `DATA_PATH` below to the exact filename.
3. Run notebook cells from top to bottom.
4. Review cleaning summaries and validation checks before interpreting charts.

**Dataset note:** Superstore copies vary in column names, date coverage, and contents. This notebook normalizes common column-name variations but raises an informative error if required fields are absent. Raw data is not overwritten.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
# If you open the notebook from the notebooks/ folder, this resolves to the project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Superstore.csv"  # Change to your actual CSV filename
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHART_DIR = PROJECT_ROOT / "outputs" / "charts"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme()
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

## 1. Load data

Most CSV versions work with `read_csv`. If you have an Excel version, save/export the transaction sheet as CSV or replace the loading line with `pd.read_excel(...)`.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Download a Sample Superstore CSV, place it in data/raw/, "
        "and update DATA_PATH to the exact filename."
    )

# Try common encodings used by CSV exports.
try:
    raw_df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
except UnicodeDecodeError:
    raw_df = pd.read_csv(DATA_PATH, encoding="latin1")

df = raw_df.copy()
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")
display(df.head())

## 2. Standardize column names

Different dataset versions may use `Order ID` or `OrderID`, and may have slightly different naming. This step standardizes formatting and maps common aliases to canonical names. Review the printed mapping if a required column is not found.

In [ ]:
def normalize_column_name(name):
    name = str(name).strip()
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", name)
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    return name.strip("_").lower()

original_columns = list(df.columns)
df.columns = [normalize_column_name(c) for c in df.columns]
print("Normalized columns:", df.columns.tolist())

ALIASES = {
    "order_id": ["order_id", "orderid", "order_number"],
    "order_date": ["order_date", "orderdate", "date"],
    "ship_date": ["ship_date", "shipdate"],
    "ship_mode": ["ship_mode", "shipmode", "shipping_mode"],
    "customer_id": ["customer_id", "customerid", "customer_number"],
    "customer_name": ["customer_name", "customername"],
    "segment": ["segment", "customer_segment"],
    "country": ["country"],
    "city": ["city"],
    "state": ["state"],
    "postal_code": ["postal_code", "postalcode", "zip_code", "zip"],
    "region": ["region"],
    "product_id": ["product_id", "productid", "product_code"],
    "category": ["category", "product_category"],
    "sub_category": ["sub_category", "subcategory", "subcat"],
    "product_name": ["product_name", "productname", "item_name"],
    "sales": ["sales", "revenue", "sales_amount", "amount"],
    "quantity": ["quantity", "qty", "units"],
    "discount": ["discount", "discount_rate"],
    "profit": ["profit", "net_profit", "profit_amount"],
}

rename_map = {}
for canonical, candidates in ALIASES.items():
    for candidate in candidates:
        if candidate in df.columns:
            if candidate != canonical:
                rename_map[candidate] = canonical
            break

df = df.rename(columns=rename_map)
print("Applied aliases:", rename_map)
print("Final columns:", df.columns.tolist())

## 3. Inspect data quality before cleaning

We record the starting row count, missing values, exact duplicates, and data types. This makes the cleaning process transparent and reproducible.

In [ ]:
starting_rows = len(df)
starting_columns = len(df.columns)

print("Shape:", df.shape)
display(df.head())
df.info()

quality_before = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_count", ascending=False)

display(quality_before)
print("Exact duplicate rows:", int(df.duplicated().sum()))

## 4. Clean fields and dates

Cleaning decisions:
- Strip whitespace from text columns.
- Parse dates safely; unparseable dates become missing and are excluded from date-based analysis.
- Convert core numeric fields to numeric values.
- Remove exact duplicate rows, not rows merely sharing an order ID.
- Keep a cleaning log so exclusions can be explained in the README.

In [ ]:
cleaning_log = []

def log_step(step, before, after, note):
    cleaning_log.append({
        "step": step,
        "rows_before": int(before),
        "rows_after": int(after),
        "rows_removed": int(before - after),
        "note": note
    })

# Trim strings, preserving missing values.
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

# Parse available date fields.
for date_col in ["order_date", "ship_date"]:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

# Convert numerical fields if present.
for numeric_col in ["sales", "profit", "quantity", "discount"]:
    if numeric_col in df.columns:
        df[numeric_col] = pd.to_numeric(df[numeric_col], errors="coerce")

before = len(df)
df = df.drop_duplicates().copy()
log_step("Remove exact duplicates", before, len(df),
         "Only rows identical across all columns are removed.")

# A usable order date is needed for trend analysis.
if "order_date" in df.columns:
    before = len(df)
    df = df.dropna(subset=["order_date"]).copy()
    log_step("Remove missing/invalid order dates", before, len(df),
             "Rows without a parseable Order Date cannot be included in time-trend analysis.")

# Sales is required for this project's primary analysis.
if "sales" not in df.columns:
    raise ValueError(
        "A Sales column was not found. Check the CSV columns and add its actual name "
        "to ALIASES before continuing."
    )

before = len(df)
df = df.dropna(subset=["sales"]).copy()
log_step("Remove missing/non-numeric sales", before, len(df),
         "Rows without a valid Sales value cannot contribute to sales KPIs.")

cleaning_log_df = pd.DataFrame(cleaning_log)
display(cleaning_log_df)
print(f"Rows at start: {starting_rows:,}; rows after core cleaning: {len(df):,}")

## 5. Validate and document unusual values

Do **not** automatically remove negative profit or discount values: these may be meaningful. Negative sales, if present, should be investigated in the context of the source dataset rather than silently deleted. We report them for review.

In [ ]:
quality_checks = {
    "rows_after_cleaning": len(df),
    "exact_duplicates_remaining": int(df.duplicated().sum()),
    "missing_order_date": int(df["order_date"].isna().sum()) if "order_date" in df.columns else None,
    "missing_sales": int(df["sales"].isna().sum()),
    "negative_sales_rows": int((df["sales"] < 0).sum()),
    "zero_sales_rows": int((df["sales"] == 0).sum()),
    "missing_profit": int(df["profit"].isna().sum()) if "profit" in df.columns else "profit column not available",
}
display(pd.Series(quality_checks, name="value").to_frame())

if "order_date" in df.columns:
    print("Order date range:", df["order_date"].min(), "to", df["order_date"].max())
print("Sales summary:")
display(df["sales"].describe().to_frame())

## 6. Calculate core KPIs

Order counts use distinct order IDs when available. If the source lacks an order identifier, we leave order count unavailable rather than treating every row as an order.

In [ ]:
total_sales = df["sales"].sum()
total_profit = df["profit"].sum() if "profit" in df.columns else None
order_count = df["order_id"].nunique() if "order_id" in df.columns else None
customer_count = df["customer_id"].nunique() if "customer_id" in df.columns else None
profit_margin = (total_profit / total_sales * 100) if total_profit is not None and total_sales != 0 else None

kpis = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Distinct Orders", "Distinct Customers", "Profit Margin (%)"],
    "Value": [total_sales, total_profit, order_count, customer_count, profit_margin]
})
display(kpis)

## 7. Create business charts

Charts are only created when the required columns exist. The outputs are saved to `outputs/charts/` for README and Power BI documentation. Check axis labels and date formats before publishing.

In [ ]:
# Monthly sales trend
if "order_date" in df.columns:
    monthly = (
        df.assign(month=df["order_date"].dt.to_period("M").dt.to_timestamp())
          .groupby("month", as_index=False)["sales"].sum()
          .sort_values("month")
    )
    if not monthly.empty:
        plt.figure(figsize=(11, 5))
        sns.lineplot(data=monthly, x="month", y="sales", marker="o")
        plt.title("Monthly Sales Trend")
        plt.xlabel("Order month")
        plt.ylabel("Sales")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(CHART_DIR / "monthly_sales_trend.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("Skipping monthly trend: order_date column unavailable.")

# Sales by category
if "category" in df.columns:
    category_sales = df.groupby("category", as_index=False)["sales"].sum().sort_values("sales", ascending=False)
    plt.figure(figsize=(9, 5))
    sns.barplot(data=category_sales, x="sales", y="category")
    plt.title("Sales by Category")
    plt.xlabel("Sales")
    plt.ylabel("Category")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "sales_by_category.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping category chart: category column unavailable.")

# Sales by region
if "region" in df.columns:
    region_sales = df.groupby("region", as_index=False)["sales"].sum().sort_values("sales", ascending=False)
    plt.figure(figsize=(9, 5))
    sns.barplot(data=region_sales, x="sales", y="region")
    plt.title("Sales by Region")
    plt.xlabel("Sales")
    plt.ylabel("Region")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "sales_by_region.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Skipping region chart: region column unavailable.")

## 8. Product profitability (if product fields exist)

This view helps distinguish high-sales products from high-profit products. Treat small groups carefully and investigate negative-profit products before making recommendations.

In [ ]:
product_group_cols = [c for c in ["product_id", "product_name", "category", "sub_category"] if c in df.columns]
if "profit" in df.columns and product_group_cols:
    product_performance = (
        df.groupby(product_group_cols, dropna=False)
          .agg(sales=("sales", "sum"), profit=("profit", "sum"))
          .reset_index()
          .sort_values("sales", ascending=False)
    )
    display(product_performance.head(10))
    display(product_performance.sort_values("profit").head(10))
else:
    print("Product profitability view requires a Profit column and at least one product/category field.")

## 9. Export cleaned data and summaries

The original raw CSV remains unchanged. Exported files can be used for review and as inputs to Power BI. Confirm that dataset sharing terms allow redistribution before committing processed data to a public repository.

In [ ]:
cleaned_path = PROCESSED_DIR / "superstore_cleaned.csv"
quality_path = PROCESSED_DIR / "data_quality_summary.csv"
cleaning_log_path = PROCESSED_DIR / "cleaning_log.csv"
kpi_path = PROCESSED_DIR / "sales_kpis.csv"

df.to_csv(cleaned_path, index=False)
quality_before.to_csv(quality_path)
cleaning_log_df.to_csv(cleaning_log_path, index=False)
kpis.to_csv(kpi_path, index=False)

print("Saved:")
for path in [cleaned_path, quality_path, cleaning_log_path, kpi_path]:
    print("-", path.relative_to(PROJECT_ROOT))

## 10. Business interpretation template

Complete these statements only after reviewing your actual outputs:
- The period with the highest sales was ______.
- The category with the highest sales was ______.
- The region with the highest sales was ______.
- The product(s) with the lowest profit were ______.
- A limitation of this analysis is ______.
- A recommendation supported by the data is ______.

**Do not claim causation from descriptive charts.** This dataset can reveal patterns and associations; it does not by itself prove why sales or profit changed.